In [ ]:
# Cell 98 (01_data_prep, cell 1): CONFIG + inspect locked discretised BN frames (READS ONLY; writes nothing)
import os, sys, json, pickle
import numpy as np, pandas as pd

assert os.environ.get('PYTHONHASHSEED') == '0', "PYTHONHASHSEED is not '0': wrong kernel, STOP"
SEED = 42
PREFIX = 'sel_'
ROOT = os.environ["CREDIT_BN_ROOT"]
PROC = ROOT + "/Data/processed/"
sys.path.insert(0, ROOT + "/shared/src")
L = lambda p: pickle.load(open(p, 'rb'))

split = L(PROC + "split_indices.pkl")
train_idx, test_idx = np.asarray(split['train_idx']), np.asarray(split['test_idx'])
y_train, y_test = np.asarray(split['y_train']), np.asarray(split['y_test'])
assert len(train_idx) == 119999 and len(test_idx) == 30000
assert round(100 * y_train.mean(), 2) == 6.68 and round(100 * y_test.mean(), 2) == 6.68

no = json.load(open(PROC + "node_orders.json"))
print("derived 7-node order:", no['derived_7_nodes_selection'])
print("KEEP_VARS:", no['KEEP_VARS'])

dfc = L(PROC + "df_clean.pkl")
pa = L(PROC + "preprocessing_artifacts.pkl")
print("\nsaved bin_edges keys:", list(pa['bin_edges'].keys()))

for nm in ['df_train', 'df_test', 'df_train_complete']:
    d = L(PROC + nm + ".pkl")
    print(f"\n--- {nm}: shape {d.shape}; index equals split: ",
          (np.array_equal(d.index.values, train_idx) if nm == 'df_train' else
           np.array_equal(d.index.values, test_idx) if nm == 'df_test' else 'n/a'))
    print("columns:", list(d.columns))
    bins = [c for c in d.columns if c.endswith('(bin)')]
    print("dtypes of bin cols:", {c: str(d[c].dtype) for c in bins[:3]}, "...")
    for c in ['M_Income', 'M_Dependents', 'Monthly Income (bin)', 'Number of Dependents (bin)', 'Default']:
        if c in d.columns:
            print(f"  {c}: NaN={int(d[c].isna().sum())}  values={d[c].astype(str).value_counts(dropna=False).sort_index().to_dict() if d[c].nunique() < 10 else 'many'}")
    if nm == 'df_train_complete':
        print("  rows:", len(d), "| index sample:", list(d.index[:5]))

dt = L(PROC + "df_test.pkl")
print("\ntest missing: Income bin NaN =", int(dt['Monthly Income (bin)'].isna().sum()),
      "| Dependents bin NaN =", int(dt['Number of Dependents (bin)'].isna().sum()))
print("bin labels, Age (bin) test:", sorted(dt['Age (bin)'].astype(str).unique()))

In [ ]:
# Cell 99 (01_data_prep, cell 2): BUILD selection frames + VALIDATE vs locked frames (READS ONLY; writes nothing)
EDGES = pa['bin_edges']
KEEP_RAW = [v.replace(' (bin)', '') for v in no['KEEP_VARS']]
EDGE_VARS = ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines']
NAT_VARS = ['Real Estate Loans', 'Number of Dependents']

def bin_natural_category(series, max_category=3):   # EXACT copy of the locked NaN-preserving function
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    def map_val(x):
        if pd.isnull(x):
            return np.nan
        return labels[int(x)] if x < max_category else labels[-1]
    return pd.Categorical(series.map(map_val), categories=labels, ordered=True)

def cut_edges(s, e):                                  # saved edges, outer edges opened to +-inf (as locked)
    ea = np.array(e, float).copy(); ea[0] = -np.inf; ea[-1] = np.inf
    return pd.cut(s, bins=ea, include_lowest=True)

def build(idx):
    raw = dfc.loc[idx]
    out = pd.DataFrame(index=raw.index)
    out['Default'] = raw['Default'].values
    for c in EDGE_VARS:
        out[c + ' (bin)'] = cut_edges(raw[c], EDGES[c])
    for c in NAT_VARS:
        out[c + ' (bin)'] = bin_natural_category(raw[c], 3)
    out['M_Income'] = raw['Monthly Income'].notna().astype(int).values       # 1 = observed
    out['M_Dependents'] = raw['Number of Dependents'].notna().astype(int).values
    return out

new_tr, new_te = build(train_idx), build(test_idx)
lock_tr, lock_te = L(PROC + "df_train.pkl"), L(PROC + "df_test.pkl")

def compare(new, lock, name):
    ok = True
    for c in new.columns:
        a, b = new[c], lock[c]
        same_idx = a.index.equals(b.index)
        if str(a.dtype) == 'category' or str(b.dtype) == 'category':
            same_lab = bool(((a.astype(object) == b.astype(object)) | (a.isna() & b.isna())).all())
            same_code = bool((a.cat.codes.values == b.cat.codes.values).all())
            same_cats = list(a.cat.categories.astype(str)) == list(b.cat.categories.astype(str))
            res = same_idx and same_lab and same_cats
            print(f"  {name:5s} {c:28s} index={same_idx} labels={same_lab} categories={same_cats} (codes={same_code})")
        else:
            res = same_idx and bool((a.values == b.values).all())
            print(f"  {name:5s} {c:28s} index={same_idx} values={res}")
        ok &= res
    return ok

ok = compare(new_tr, lock_tr, 'train') & compare(new_te, lock_te, 'test')
print("\nALL SHARED COLUMNS IDENTICAL TO LOCKED FRAMES:", ok)
assert ok, "STOP: mismatch with locked discretised frames"

# complete cases (rows with Income AND Dependents observed) vs locked df_train_complete
cc_mask = (new_tr['M_Income'] == 1) & (new_tr['M_Dependents'] == 1)
lock_cc = L(PROC + "df_train_complete.pkl")
print("\ncomplete cases:", int(cc_mask.sum()), "| index equals locked df_train_complete:", new_tr.index[cc_mask].equals(lock_cc.index))
assert cc_mask.sum() == 96251 and new_tr.index[cc_mask].equals(lock_cc.index)

# asserts from the spec
for nm, d in [('train', new_tr), ('test', new_te)]:
    print(f"{nm}: rows={len(d)}, default rate={d['Default'].mean():.4%}, Income missing={int((d['M_Income']==0).sum())}, Dependents missing={int((d['M_Dependents']==0).sum())}")
    assert round(100 * d['Default'].mean(), 2) == 6.68
    assert not ((d['M_Dependents'] == 0) & (d['M_Income'] == 1)).any(), "a Dependents-missing row is not Income-missing"
assert (new_te['M_Income'] == 0).sum() == 5983 and (new_te['M_Dependents'] == 0).sum() == 777
print("every Dependents-missing row is also Income-missing (train and test): True")
print("\nPART A VALIDATION PASSED (nothing saved yet)")

In [ ]:
# Cell 100 (01_data_prep, cell 3): SAVE Part A outputs via safe_save (sel_ prefix). Needs Cell 99 variables in the kernel.
# WRITES (new): Data/processed/sel_train_full.pkl, sel_train_cc.pkl, sel_test.pkl, sel_state_names.json, sel_node_order.json
from safe_save import safe_save

NODES = no['derived_7_nodes_selection'] + ['M_Income', 'M_Dependents']
assert len(NODES) == 9

def frame(d):
    out = d[['Default'] + [v for v in no['KEEP_VARS']] + ['M_Income', 'M_Dependents']].copy()
    return out

sel_train_full, sel_test = frame(new_tr), frame(new_te)
sel_train_cc = sel_train_full.loc[cc_mask.values].copy()
assert sel_train_full.shape == (119999, 9) and sel_test.shape == (30000, 9) and sel_train_cc.shape == (96251, 9)
assert sel_train_cc.index.equals(lock_cc.index)
assert sel_train_cc.drop(columns=['M_Income', 'M_Dependents']).notna().all().all()

# state names from train UNION test; observed states only, ordered by the locked categories
state_names = {}
for c in NODES:
    s = pd.concat([sel_train_full[c], sel_test[c]])
    if str(s.dtype) == 'category':
        cats = [str(x) for x in s.cat.categories]
        present = set(s.dropna().astype(str))
        state_names[c] = [x for x in cats if x in present]
    else:
        state_names[c] = sorted(int(x) for x in s.dropna().unique())
print({k: len(v) for k, v in state_names.items()})
assert all(len(v) >= 2 for v in state_names.values())

node_order = dict(derived_7_nodes_selection=no['derived_7_nodes_selection'], all_9_nodes=NODES,
                  indicator_edges=[['M_Income', 'Monthly Income (bin)'], ['M_Dependents', 'Number of Dependents (bin)']])

for obj, name in [(sel_train_full, "sel_train_full.pkl"), (sel_train_cc, "sel_train_cc.pkl"), (sel_test, "sel_test.pkl"),
                  (state_names, "sel_state_names.json"), (node_order, "sel_node_order.json")]:
    path, h = safe_save(obj, PROC + name)
    print("saved:", name, h[:12])
print("shapes:", sel_train_full.shape, sel_train_cc.shape, sel_test.shape)

In [ ]:
# Cell 101 (01_data_prep, cell 4): write shared/src/structure.py (new file, exclusive-create) and test that it imports. Runs NO learning.
SRC = '''"""structure: multi-restart hill-climbing + BIC structure learning.

Wrapped, with no logic changes, from the locked run (old notebook cell 27, recovered from the kernel history):
  - start DAG: random.Random(seed) shuffles a COPY of node_order; each forward-only pair (i<j) gets an edge with prob edge_density
  - HillClimbSearch(data).estimate(scoring_method=BIC(data), start_dag=start, show_progress=False)
  - score = BIC(data).score(learned_dag); best = highest score; ties keep the earliest restart (stable sort)
The result depends on the ORDER of node_order and on PYTHONHASHSEED=0 (set before Python starts).
"""
import os, random, time


def random_start_dag(nodes, rng, edge_prob=0.15):
    from pgmpy.base import DAG
    order = list(nodes)
    rng.shuffle(order)
    dag = DAG()
    dag.add_nodes_from(order)
    for i in range(len(order)):
        for j in range(i + 1, len(order)):
            if rng.random() < edge_prob:
                dag.add_edge(order[i], order[j])
    return dag


def learn_structure(data, node_order, seeds=range(42, 57), edge_density=0.15, verbose=True):
    """Return dict(best_score, best_dag, best_seed, best_edges, results=[(score, dag, seed), ...], runtime_s)."""
    assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED must be '0' before Python starts"
    from pgmpy.estimators import HillClimbSearch, BIC
    assert list(data.columns) == list(node_order), "data column order must equal node_order"
    t0 = time.time()
    bic = BIC(data)
    results = []
    for seed in seeds:
        rng = random.Random(seed)
        start = random_start_dag(node_order, rng, edge_density)
        learned = HillClimbSearch(data).estimate(scoring_method=bic, start_dag=start, show_progress=False)
        score = bic.score(learned)
        results.append((score, learned, seed))
        if verbose:
            print(f"seed {seed}: BIC = {score:.2f}, edges = {len(learned.edges())}  ({time.time() - t0:.0f}s elapsed)")
    ranked = sorted(results, key=lambda x: x[0], reverse=True)
    best_score, best_dag, best_seed = ranked[0]
    return dict(best_score=best_score, best_dag=best_dag, best_seed=best_seed,
                best_edges=sorted(map(tuple, best_dag.edges())), results=results, runtime_s=time.time() - t0)
'''
dest = ROOT + "/shared/src/structure.py"
assert not os.path.exists(dest), "structure.py already exists: STOP"
with open(dest, "xb") as fh:
    fh.write(SRC.encode())
print("written:", dest)

import importlib, structure
importlib.reload(structure)
print("import OK; functions:", [n for n in dir(structure) if n in ("learn_structure", "random_start_dag")])

In [ ]:
# Cell 102 (01_data_prep, cell 5): Part B VALIDATION on the LOCKED 11-node frame (READS ONLY; writes nothing)
import importlib, structure
importlib.reload(structure)

dtc = L(PROC + "df_train_complete.pkl")
orig_nodes = no['original_11_nodes']
assert dtc.shape == (96251, 11), dtc.shape
assert list(dtc.columns) == list(orig_nodes), "column order differs from original_11_nodes: STOP"
print("frame:", dtc.shape, "| node order OK | PYTHONHASHSEED =", os.environ['PYTHONHASHSEED'])

res = structure.learn_structure(dtc, orig_nodes, seeds=range(42, 57), edge_density=0.15)

locked = L(ROOT + "/full_variable/models/bn_final_model_iteration79.pkl")
IND = {('M_Income', 'Monthly Income (bin)'), ('M_Dependents', 'Number of Dependents (bin)')}
locked_edges = sorted(tuple(e) for e in locked['model_edges'] if tuple(e) not in IND)
new_edges = res['best_edges']
print(f"\nlocked learned edges: {len(locked_edges)} (24 minus 2 indicator edges) | new: {len(new_edges)}")
only_locked = sorted(set(locked_edges) - set(new_edges))
only_new = sorted(set(new_edges) - set(locked_edges))
print("in locked but not new:", only_locked)
print("in new but not locked:", only_new)
print("EDGE SETS IDENTICAL:", not only_locked and not only_new)

print(f"\nbest seed: {res['best_seed']} | final (best) BIC: {res['best_score']:.4f} | runtime: {res['runtime_s']:.0f}s")
from pgmpy.estimators import BIC
from pgmpy.base import DAG
bic = BIC(dtc)
d_locked = DAG(); d_locked.add_nodes_from(orig_nodes); d_locked.add_edges_from(locked_edges)
print(f"BIC of the locked 22-edge structure (same scorer): {bic.score(d_locked):.4f}")
print("BIC per seed:", {s: round(sc, 2) for sc, _, s in res['results']})